In [10]:
"""
Fair nonlinear functional time series simulation:
FPCA vs FAE

Fair comparison:
- same latent dimension for both methods
- same forecasting model for both methods: VAR(1)
- FAE is trained for reconstruction only (plus optional smoothness penalty)
- nonlinear data-generating observation map favors nonlinear representation learning

Main truth:
    z_t = a + A z_{t-1} + eps_t
    b_t = W_lin z_t + W_nl q(z_t) + c
    x_t = Psi b_t + eta_t

where q(z_t) contains nonlinear features such as:
    z1^2, z2^2, z3^2, z1*z2, z1*z3, z2*z3, tanh(z1), tanh(z2), tanh(z3)

Idea:
- latent dynamics are linear/stable, so VAR(1) is fair for both
- observation map is nonlinear, so FPCA's linear scores are distorted
- FAE can learn a better low-dimensional nonlinear representation
- then the SAME VAR(1) on FAE latent codes can forecast better
"""

from __future__ import annotations

import copy
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch import optim


# ============================================================
# 0) Utilities
# ============================================================

def set_seed(seed: int = 123) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def rel_mse(y_hat: torch.Tensor, y_true: torch.Tensor, eps: float = 1e-12) -> float:
    mse = torch.mean((y_hat - y_true) ** 2)
    var = torch.var(y_true, unbiased=False)
    return float((mse / var.clamp_min(eps)).detach().cpu())


def mse(y_hat: torch.Tensor, y_true: torch.Tensor) -> float:
    return float(torch.mean((y_hat - y_true) ** 2).detach().cpu())


def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    if horizon < 1:
        raise ValueError("horizon must be >= 1")
    if X.shape[0] <= horizon:
        raise ValueError("Need more observations than horizon.")
    return X[:-horizon], X[-horizon:]

def print_report_table(title: str, rows: Dict[str, Dict[str, float]]) -> None:
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))
    header = f"{'Method':<18} {'relMSE vs CLEAN':>18} {'relMSE vs NOISY':>18}"
    print(header)
    print("-" * len(header))
    for method, vals in rows.items():
        a = vals.get("relMSE_vs_clean", float("nan"))
        b = vals.get("relMSE_vs_noisy", float("nan"))
        print(f"{method:<18} {a:18.6f} {b:18.6f}")


# ============================================================
# 1) Basis builder
# ============================================================

class BasisFCBuilder:
    def __init__(self, n_basis: int = 20, bspline_degree: int = 3):
        self.n_basis = int(n_basis)
        self.bspline_degree = int(bspline_degree)

    def build(self, tpts: torch.Tensor) -> torch.Tensor:
        return self._build_bspline(tpts, self.bspline_degree)

    def _build_bspline(self, tpts: torch.Tensor, degree: int) -> torch.Tensor:
        t = tpts.flatten()
        t_min = t.min()
        t_max = t.max()
        denom = (t_max - t_min).clamp_min(1e-8)
        tau = (t - t_min) / denom
        tau_np = tau.detach().cpu().numpy()

        n_time = tau_np.shape[0]
        n_basis = self.n_basis
        p = degree

        if n_basis < p + 1:
            raise ValueError(f"n_basis must be >= degree+1={p+1}")

        n_int = max(n_basis - p - 1, 0)
        if n_int > 0:
            interior = np.linspace(0.0, 1.0, n_int + 2)[1:-1]
            knots = np.concatenate((np.zeros(p + 1), interior, np.ones(p + 1)))
        else:
            knots = np.concatenate((np.zeros(p + 1), np.ones(p + 1)))

        N = np.zeros((n_basis, n_time), dtype=np.float64)
        for i in range(n_basis):
            left = knots[i]
            right = knots[i + 1]
            N[i, :] = np.where((tau_np >= left) & (tau_np < right), 1.0, 0.0)
        N[-1, tau_np == 1.0] = 1.0

        for k in range(1, p + 1):
            N_next = np.zeros_like(N)
            for i in range(n_basis):
                denom_left = knots[i + k] - knots[i]
                if denom_left > 0:
                    c_left = (tau_np - knots[i]) / denom_left
                    left_part = c_left * N[i, :]
                else:
                    left_part = 0.0

                denom_right = (knots[i + k + 1] - knots[i + 1]) if (i + 1) < n_basis else 0.0
                if denom_right > 0 and (i + 1) < n_basis:
                    c_right = (knots[i + k + 1] - tau_np) / denom_right
                    right_part = c_right * N[i + 1, :]
                else:
                    right_part = 0.0

                N_next[i, :] = left_part + right_part
            N = N_next

        return torch.tensor(N.T, dtype=torch.float32, device=tpts.device)


# ============================================================
# 2) Latent linear VAR(1) truth
# ============================================================

def spectral_radius(A: np.ndarray) -> float:
    return float(np.max(np.abs(np.linalg.eigvals(A))))


def stabilize_A(A: np.ndarray, target_rho: float = 0.75) -> np.ndarray:
    rho = spectral_radius(A)
    if rho <= target_rho or rho <= 1e-12:
        return A
    return (target_rho / rho) * A


def generate_latent_var1(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    target_rho: float = 0.70,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    z_t = a + A z_{t-1} + eps_t
    """
    rng = np.random.default_rng(seed)

    A = rng.normal(0.0, 0.16, size=(d, d))
    A = stabilize_A(A, target_rho=target_rho)

    a = rng.normal(0.0, 0.08, size=d)

    Z = np.zeros((T, d), dtype=float)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma)

    for t in range(1, T):
        eps = rng.multivariate_normal(np.zeros(d), Sigma)
        Z[t] = a + A @ Z[t - 1] + eps

    return Z, a, A

# ============================================================
# 3) Stronger nonlinear observation truth
# ============================================================

def build_true_basis(tpts: torch.Tensor, n_basis: int, degree: int) -> torch.Tensor:
    builder = BasisFCBuilder(n_basis=n_basis, bspline_degree=degree)
    return builder.build(tpts)


def nonlinear_lift(Z: torch.Tensor) -> torch.Tensor:
    """
    Smooth, structured nonlinear lift.
    Curved enough to hurt FPCA, but still learnable by FAE.
    """
    z1 = Z[:, 0:1]
    z2 = Z[:, 1:2]
    z3 = Z[:, 2:3]

    u1 = z1 + 0.45 * (z2 ** 2) + 0.20 * torch.sin(1.2 * z3)
    u2 = z2 + 0.35 * torch.sin(1.3 * z1) + 0.25 * (z3 ** 2)
    u3 = z3 + 0.40 * (z1 * z2) + 0.20 * torch.tanh(1.1 * z2)

    feats = [
        u1, u2, u3,
        u1 * u2,
        u1 * u3,
        u2 * u3,
        u1 ** 2,
        u2 ** 2,
        u3 ** 2,
        torch.tanh(u1),
        torch.tanh(u2),
        torch.tanh(u3),
    ]
    return torch.cat(feats, dim=1)


def nonlinear_coef_decoder_truth(
    Z: torch.Tensor,
    Psi: torch.Tensor,
    seed: int = 123,
    lift_scale: float = 0.40,
    nonlinear_gain: float = 1.35,
    bias_scale: float = 0.04,
    noise_sd: float = 0.002,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Structured nonlinear coefficient map.
    """
    torch.manual_seed(seed)

    q = nonlinear_lift(Z)
    qdim = q.shape[1]
    M = Psi.shape[1]
    hidden = 10

    W1 = torch.randn(qdim, hidden, dtype=torch.float32) * lift_scale
    c1 = torch.randn(hidden, dtype=torch.float32) * bias_scale

    W2 = torch.randn(hidden, M, dtype=torch.float32) * lift_scale
    c2 = torch.randn(M, dtype=torch.float32) * bias_scale

    H = torch.tanh(q @ W1 + c1.unsqueeze(0))
    Bcoef = nonlinear_gain * (H @ W2) + c2.unsqueeze(0)

    X_clean = Bcoef @ Psi.T
    X_noisy = X_clean + noise_sd * torch.randn_like(X_clean)

    return Bcoef, X_clean, X_noisy


# ============================================================
# 4) Simulation config
# ============================================================

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 320
    P: int = 100
    d: int = 3
    Sigma_scale: float = 0.055
    target_rho: float = 0.72

    true_n_basis: int = 15
    true_bspline_degree: int = 3

    lift_scale: float = 0.55
    nonlinear_gain: float = 1.80
    bias_scale: float = 0.05

    meas_noise_sd: float = 0.002


def simulate_nonlinear_functional_ts(cfg: SimCfg) -> Dict[str, object]:
    set_seed(cfg.seed)

    u = np.linspace(0.0, 1.0, cfg.P).astype(float)
    tpts = torch.tensor(u, dtype=torch.float32)

    Sigma = (cfg.Sigma_scale ** 2) * np.eye(cfg.d)
    Z_np, a_true, A_true = generate_latent_var1(
        T=cfg.T,
        d=cfg.d,
        Sigma=Sigma,
        seed=cfg.seed,
        target_rho=cfg.target_rho,
    )
    Z = torch.tensor(Z_np, dtype=torch.float32)

    Psi = build_true_basis(
        tpts=tpts,
        n_basis=cfg.true_n_basis,
        degree=cfg.true_bspline_degree,
    )

    Bcoef, X_clean, X_noisy = nonlinear_coef_decoder_truth(
        Z=Z,
        Psi=Psi,
        seed=cfg.seed,
        lift_scale=cfg.lift_scale,
        nonlinear_gain=cfg.nonlinear_gain,
        bias_scale=cfg.bias_scale,
        noise_sd=cfg.meas_noise_sd,
    )

    return {
        "u": u,
        "tpts": tpts,
        "Z_true": Z,
        "a_true": a_true,
        "A_true": A_true,
        "Psi_true": Psi,
        "Bcoef_true": Bcoef,
        "X_clean": X_clean,
        "X_noisy": X_noisy,
    }


# ============================================================
# 5) FPCA baseline
# ============================================================

@torch.no_grad()
def centered_weighted_fpca_fit(X_train: torch.Tensor, w: torch.Tensor, K: int):
    mu = X_train.mean(dim=0)
    Xc = X_train - mu.unsqueeze(0)

    sw = torch.sqrt(w).view(1, -1)
    Xw = Xc * sw

    U, S, Vh = torch.linalg.svd(Xw, full_matrices=False)
    V = Vh.transpose(0, 1)

    K = min(K, V.shape[1])
    phi = (V[:, :K] / sw.flatten()[:, None]).T

    for k in range(K):
        nrm = torch.sqrt(torch.sum(phi[k] * phi[k] * w))
        phi[k] = phi[k] / nrm.clamp_min(1e-12)

    scores = (Xc * w.view(1, -1)) @ phi.T
    return mu, phi, scores


@torch.no_grad()
def fpca_reconstruct_from_trainfit(X_train: torch.Tensor, X_eval: torch.Tensor, u: np.ndarray, K: int):
    w = trapezoid_weights(u)
    mu, phi, scores_train = centered_weighted_fpca_fit(X_train, w, K)
    scores_eval = ((X_eval - mu.unsqueeze(0)) * w.view(1, -1)) @ phi.T
    Xhat_eval = mu.unsqueeze(0) + scores_eval @ phi
    return Xhat_eval, mu, phi, scores_train


# ============================================================
# 6) Shared VAR(1) forecasting model
# ============================================================

def fit_var1(H: np.ndarray, ridge: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    T, K = H.shape
    if T < 2:
        raise ValueError("Need at least 2 observations for VAR(1).")

    X = H[:-1, :]
    Y = H[1:, :]
    X_aug = np.hstack([np.ones((T - 1, 1)), X])

    XtX = X_aug.T @ X_aug
    B = np.linalg.solve(XtX + ridge * np.eye(XtX.shape[0]), X_aug.T @ Y)

    a = B[0, :]
    A = B[1:, :].T
    return a, A


def forecast_var1(h_last: np.ndarray, a: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    out = np.zeros((steps, h_last.shape[0]), dtype=float)
    h = h_last.copy()

    for i in range(steps):
        h = a + A @ h
        out[i, :] = h

    return out


@torch.no_grad()
def fpca_var_forecast(X_train: torch.Tensor, u: np.ndarray, K: int, steps: int, ridge: float = 1e-6):
    w = trapezoid_weights(u)
    mu, phi, scores = centered_weighted_fpca_fit(X_train, w, K)

    H = scores.detach().cpu().numpy()
    a, A = fit_var1(H, ridge=ridge)
    Hf = forecast_var1(H[-1], a, A, steps=steps)

    Hf_t = torch.tensor(Hf, dtype=torch.float32)
    X_fore = mu.unsqueeze(0) + Hf_t @ phi
    return X_fore


# ============================================================
# 7) FAE model
# ============================================================

class FAE(nn.Module):
    def __init__(
        self,
        n_basis_project: int,
        n_rep: int,
        n_basis_revert: int,
        hidden1: int = 96,
        hidden2: int = 96,
        init_weight_sd: Optional[float] = 0.04,
    ):
        super().__init__()

        self.fc1 = nn.Linear(n_basis_project, hidden1)
        self.fc2 = nn.Linear(hidden1, n_rep)

        self.fc3 = nn.Linear(n_rep, hidden2)
        self.fc4 = nn.Linear(hidden2, n_basis_revert)

        self.activation = nn.Tanh()

        if init_weight_sd is not None:
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.normal_(m.weight, mean=0.0, std=init_weight_sd)
                    nn.init.zeros_(m.bias)

    def project(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc: torch.Tensor) -> torch.Tensor:
        t = tpts.flatten()
        dt = t[1:] - t[:-1]
        zero = torch.zeros(1, device=x.device, dtype=x.dtype)
        W = 0.5 * torch.cat([zero, dt]) + 0.5 * torch.cat([dt, zero])

        if basis_fc.shape[0] == x.shape[1]:
            B = basis_fc
        elif basis_fc.shape[1] == x.shape[1]:
            B = basis_fc.T
        else:
            raise RuntimeError("basis_fc has incompatible shape")

        return (x * W) @ B

    def revert(self, coef: torch.Tensor, basis_fc: torch.Tensor) -> torch.Tensor:
        if basis_fc.shape[1] == coef.shape[1]:
            return coef @ basis_fc.T
        elif basis_fc.shape[0] == coef.shape[1]:
            return coef @ basis_fc
        else:
            raise RuntimeError("basis_fc has incompatible shape")

    def encode(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc_project: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.project(x, tpts, basis_fc_project)
        h1 = self.activation(self.fc1(feat))
        rep = self.fc2(h1)
        return feat, rep

    def decode_from_rep(self, rep: torch.Tensor, basis_fc_revert: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h2 = self.activation(self.fc3(rep))
        coef = self.fc4(h2)
        x_hat = self.revert(coef, basis_fc_revert)
        return x_hat, coef

    def forward(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc_project: torch.Tensor, basis_fc_revert: torch.Tensor):
        feat, rep = self.encode(x, tpts, basis_fc_project)
        x_hat, coef = self.decode_from_rep(rep, basis_fc_revert)
        return x_hat, rep, coef


def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    if coef.shape[1] < 3:
        return torch.tensor(0.0, device=coef.device, dtype=coef.dtype)
    delta = coef[:, 2:] - 2 * coef[:, 1:-1] + coef[:, :-2]
    return torch.mean(torch.sum(delta ** 2, dim=1))


@dataclass
class FaeCfg:
    seed: int = 777
    device: str = "cpu"

    n_basis_project: int = 30
    n_basis_revert: int = 30
    bspline_degree: int = 3

    n_rep: int = 3
    hidden1: int = 96
    hidden2: int = 96
    init_weight_sd: float = 0.04

    epochs: int = 1800
    batch_size: int = 16
    lr: float = 7e-4
    weight_decay: float = 1e-6
    split_rate: float = 0.85
    log_every: int = 200

    smooth_lambda: float = 5e-5


def train_fae_on_train(
    Xn_train: torch.Tensor,
    Xc_train: torch.Tensor,
    tpts: torch.Tensor,
    cfg: FaeCfg,
):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    builder_proj = BasisFCBuilder(n_basis=cfg.n_basis_project, bspline_degree=cfg.bspline_degree)
    builder_rev = BasisFCBuilder(n_basis=cfg.n_basis_revert, bspline_degree=cfg.bspline_degree)

    Bp = builder_proj.build(tpts.to(device)).to(device)
    Br = builder_rev.build(tpts.to(device)).to(device)
    tpts_d = tpts.to(device).float()

    loss_fn = nn.MSELoss()

    n = Xn_train.shape[0]
    n_tr = max(8, int(cfg.split_rate * n))
    n_tr = min(n_tr, n - 1)

    Xn_tr = Xn_train[:n_tr].float().to(device)
    Xc_tr = Xc_train[:n_tr].float().to(device)
    Xn_va = Xn_train[n_tr:].float().to(device)
    Xc_va = Xc_train[n_tr:].float().to(device)

    model = FAE(
        n_basis_project=cfg.n_basis_project,
        n_rep=cfg.n_rep,
        n_basis_revert=cfg.n_basis_revert,
        hidden1=cfg.hidden1,
        hidden2=cfg.hidden2,
        init_weight_sd=cfg.init_weight_sd,
    ).to(device)

    opt = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_state = None
    best_val = float("inf")

    for ep in range(1, cfg.epochs + 1):
        model.train()

        for start in range(0, Xn_tr.shape[0], cfg.batch_size):
            xb_in = Xn_tr[start:start + cfg.batch_size]
            xb_tg = Xc_tr[start:start + cfg.batch_size]

            opt.zero_grad()
            xhat, _, coef = model(xb_in, tpts_d, Bp, Br)
            loss = loss_fn(xhat, xb_tg) + cfg.smooth_lambda * diff_penalty(coef)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            va_hat, _, coef_va = model(Xn_va, tpts_d, Bp, Br)
            va_loss = loss_fn(va_hat, Xc_va) + cfg.smooth_lambda * diff_penalty(coef_va)

        if va_loss < best_val:
            best_val = float(va_loss)
            best_state = copy.deepcopy(model.state_dict())

        if cfg.log_every and (ep % cfg.log_every == 0):
            print(f"[FAE] epoch {ep:4d} | val clean objective = {float(va_loss):.6e}")

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, tpts_d.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()


@torch.no_grad()
def fae_reconstruct(model: FAE, X: torch.Tensor, tpts: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor):
    device = next(model.parameters()).device
    X = X.to(device).float()
    tpts = tpts.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)

    Xhat, H, coef = model(X, tpts, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu(), coef.detach().cpu()


@torch.no_grad()
def fae_var_forecast(model: FAE, Xn_train: torch.Tensor, steps: int, tpts: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor, ridge: float = 1e-6):
    _, H_train, _ = fae_reconstruct(model, Xn_train, tpts, Bp, Br)

    H_np = H_train.numpy()
    a, A = fit_var1(H_np, ridge=ridge)
    Hf = forecast_var1(H_np[-1], a, A, steps=steps)

    device = next(model.parameters()).device
    Hf_t = torch.tensor(Hf, dtype=torch.float32, device=device)
    X_fore, _ = model.decode_from_rep(Hf_t, Br.to(device))
    return X_fore.detach().cpu()


# ============================================================
# 8) Run one experiment
# ============================================================

@dataclass
class RunCfg:
    horizon: int = 5
    fpca_K: int = 3
    var_ridge: float = 1e-6


def run_one_experiment(sim_cfg: SimCfg, run_cfg: RunCfg, fae_cfg: FaeCfg) -> Dict[str, float]:
    if run_cfg.fpca_K != fae_cfg.n_rep:
        raise ValueError("For a fair comparison, set run_cfg.fpca_K == fae_cfg.n_rep")

    sim = simulate_nonlinear_functional_ts(sim_cfg)

    u = sim["u"]
    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    X_fpca_recon_train, _, _, _ = fpca_reconstruct_from_trainfit(
        X_train=Xn_train,
        X_eval=Xn_train,
        u=u,
        K=run_cfg.fpca_K,
    )

    fae_model, tpts_fae, Bp_fae, Br_fae = train_fae_on_train(
        Xn_train=Xn_train,
        Xc_train=Xc_train,
        tpts=tpts,
        cfg=fae_cfg,
    )
    X_fae_recon_train, _, _ = fae_reconstruct(fae_model, Xn_train, tpts_fae, Bp_fae, Br_fae)

    recon_rows = {
        "FPCA": {
            "relMSE_vs_clean": rel_mse(X_fpca_recon_train, Xc_train),
            "relMSE_vs_noisy": rel_mse(X_fpca_recon_train, Xn_train),
        },
        "FAE": {
            "relMSE_vs_clean": rel_mse(X_fae_recon_train, Xc_train),
            "relMSE_vs_noisy": rel_mse(X_fae_recon_train, Xn_train),
        },
    }
    print_report_table("REPORT 1 -- Reconstruction on TRAIN", recon_rows)

    X_fpca_fore = fpca_var_forecast(
        X_train=Xn_train,
        u=u,
        K=run_cfg.fpca_K,
        steps=run_cfg.horizon,
        ridge=run_cfg.var_ridge,
    )

    X_fae_fore = fae_var_forecast(
        model=fae_model,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        tpts=tpts_fae,
        Bp=Bp_fae,
        Br=Br_fae,
        ridge=run_cfg.var_ridge,
    )

    fore_rows = {
        "FPCA+VAR": {
            "relMSE_vs_clean": rel_mse(X_fpca_fore, Xc_test),
            "relMSE_vs_noisy": rel_mse(X_fpca_fore, Xn_test),
        },
        "FAE+VAR": {
            "relMSE_vs_clean": rel_mse(X_fae_fore, Xc_test),
            "relMSE_vs_noisy": rel_mse(X_fae_fore, Xn_test),
        },
    }
    print_report_table(f"REPORT 2 -- Forecast on TEST horizon (H={run_cfg.horizon})", fore_rows)

    return {
        "fpca_recon_relMSE_vs_clean": rel_mse(X_fpca_recon_train, Xc_train),
        "fpca_recon_relMSE_vs_noisy": rel_mse(X_fpca_recon_train, Xn_train),
        "fae_recon_relMSE_vs_clean": rel_mse(X_fae_recon_train, Xc_train),
        "fae_recon_relMSE_vs_noisy": rel_mse(X_fae_recon_train, Xn_train),
        "fpca_fore_relMSE_vs_clean": rel_mse(X_fpca_fore, Xc_test),
        "fpca_fore_relMSE_vs_noisy": rel_mse(X_fpca_fore, Xn_test),
        "fae_fore_relMSE_vs_clean": rel_mse(X_fae_fore, Xc_test),
        "fae_fore_relMSE_vs_noisy": rel_mse(X_fae_fore, Xn_test),
    }


# ============================================================
# 9) Batch experiment
# ============================================================

def run_many(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 20,
    seed_base: int = 5000,
    verbose: bool = False,
) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []

    old_log = fae_cfg.log_every
    if not verbose:
        fae_cfg.log_every = 0

    try:
        for rep in range(1, n_reps + 1):
            sim_cfg_rep = copy.deepcopy(sim_cfg)
            fae_cfg_rep = copy.deepcopy(fae_cfg)

            sim_cfg_rep.seed = seed_base + rep
            fae_cfg_rep.seed = seed_base + 1000 + rep

            res = run_one_experiment(sim_cfg_rep, run_cfg, fae_cfg_rep)
            res["rep"] = rep

            res["fae_wins_recon_clean"] = int(
                res["fae_recon_relMSE_vs_clean"] < res["fpca_recon_relMSE_vs_clean"]
            )
            res["fae_wins_fore_clean"] = int(
                res["fae_fore_relMSE_vs_clean"] < res["fpca_fore_relMSE_vs_clean"]
            )
            res["fae_wins_both_clean"] = int(
                (res["fae_recon_relMSE_vs_clean"] < res["fpca_recon_relMSE_vs_clean"]) and
                (res["fae_fore_relMSE_vs_clean"] < res["fpca_fore_relMSE_vs_clean"])
            )

            rows.append(res)

        return pd.DataFrame(rows)

    finally:
        fae_cfg.log_every = old_log


# ============================================================
# 10) Main
# ============================================================

if __name__ == "__main__":
    sim_cfg = SimCfg(
        seed=123,
        T=320,
        P=100,
        d=3,
        Sigma_scale=0.045,
        target_rho=0.82,
        true_n_basis=14,
        true_bspline_degree=3,
        lift_scale=0.40,
        nonlinear_gain=1.35,
        bias_scale=0.04,
        meas_noise_sd=0.002,
    )

    fae_cfg = FaeCfg(
        seed=777,
        device="cpu",
        n_basis_project=40,
        n_basis_revert=40,
        bspline_degree=3,
        n_rep=3,
        hidden1=160,
        hidden2=160,
        init_weight_sd=0.02,
        epochs=3000,
        batch_size=16,
        lr=8e-4,
        weight_decay=0.0,
        split_rate=0.85,
        log_every=200,
        smooth_lambda=0.0,
    )

    run_cfg = RunCfg(
        horizon=2,
        fpca_K=fae_cfg.n_rep,
        var_ridge=1e-6,
    )

    print("\nRunning one experiment...")
    one_res = run_one_experiment(sim_cfg, run_cfg, fae_cfg)

    print("\nSingle-run summary:")
    for k, v in one_res.items():
        print(f"{k}: {v:.6f}")

    print("\nRunning multiple replications...")
    df = run_many(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        seed_base=5000,
        verbose=False,
    )

    print("\nPer-replication results:")
    print(df.round(4))

    print("\nAverage results:")
    print(df.mean(numeric_only=True).round(4))

    print("\nWin rates:")
    print("FAE wins reconstruction (clean):", df["fae_wins_recon_clean"].mean().round(4))
    print("FAE wins forecast      (clean):", df["fae_wins_fore_clean"].mean().round(4))
    print("FAE wins both          (clean):", df["fae_wins_both_clean"].mean().round(4))


Running one experiment...
[FAE] epoch  200 | val clean objective = 9.964087e-06
[FAE] epoch  400 | val clean objective = 9.819882e-06
[FAE] epoch  600 | val clean objective = 9.441608e-06
[FAE] epoch  800 | val clean objective = 9.556448e-06
[FAE] epoch 1000 | val clean objective = 1.127089e-05
[FAE] epoch 1200 | val clean objective = 1.013303e-05
[FAE] epoch 1400 | val clean objective = 9.585488e-06
[FAE] epoch 1600 | val clean objective = 9.991184e-06
[FAE] epoch 1800 | val clean objective = 1.005026e-05
[FAE] epoch 2000 | val clean objective = 1.007946e-05
[FAE] epoch 2200 | val clean objective = 1.007925e-05
[FAE] epoch 2400 | val clean objective = 1.003024e-05
[FAE] epoch 2600 | val clean objective = 9.994092e-06
[FAE] epoch 2800 | val clean objective = 9.575486e-06
[FAE] epoch 3000 | val clean objective = 9.907506e-06

REPORT 1 -- Reconstruction on TRAIN
Method                relMSE vs CLEAN    relMSE vs NOISY
--------------------------------------------------------
FPCA        

In [11]:
df.to_excel("test.xlsx")